# KPI potentiel Parkshare par commune

Objectif:
- lire `datasetfinal.csv`,
- calculer un score de potentiel commercial par commune,
- exporter un CSV KPI exploitable dans le dashboard.

Le score privilegie:
- la demande potentielle (population, vehicules),
- l'opportunite copropriete (nombre de copros et grandes copros),
- le gisement de stationnement (lots/points).


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

BASE_DIR = Path.cwd()
INPUT_PATH = BASE_DIR / "Sources_clean" / "datasetfinal.csv"
OUTPUT_PATH = BASE_DIR / "Sources_clean" / "kpi_potentiel_communes.csv"

print("INPUT:", INPUT_PATH)
print("OUTPUT:", OUTPUT_PATH)

In [ ]:
df = pd.read_csv(INPUT_PATH, low_memory=False)
print(f"Shape: {df.shape}")
print("Colonnes:", df.columns.tolist())
df.head(3)

In [ ]:
numeric_cols = [
    "nb_vehicules",
    "nb_vp_elec",
    "population_totale",
    "residences_principales",
    "pop_15_29",
    "logements_vacants",
    "logements_locatifs",
    "nb_copros",
    "nb_logements",
    "copros_51_200",
    "copros_200_plus",
    "part_grandes_copros",
    "total_lots_stationnement",
    "moyenne_lots_par_point",
    "nb_points_parking",
    "lat",
    "long",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

missing_ratio = (df[numeric_cols].isna().mean() * 100).sort_values(ascending=False)
missing_ratio.head(10)

## Feature engineering

On calcule des variables plus proches du besoin metier Parkshare.

In [ ]:
df["grandes_copros_nb"] = df[["copros_51_200", "copros_200_plus"]].fillna(0).sum(axis=1)
df["motorisation_par_habitant"] = df["nb_vehicules"] / df["population_totale"].replace({0: np.nan})
df["lots_stationnement_par_habitant"] = df["total_lots_stationnement"] / df["population_totale"].replace({0: np.nan})
df["lots_stationnement_par_copro"] = df["total_lots_stationnement"] / df["nb_copros"].replace({0: np.nan})

# bornes defensives contre les outliers extremement rares
for c in ["motorisation_par_habitant", "lots_stationnement_par_habitant", "lots_stationnement_par_copro"]:
    q_low = df[c].quantile(0.01)
    q_high = df[c].quantile(0.99)
    df[c] = df[c].clip(lower=q_low, upper=q_high)

df[[
    "grandes_copros_nb",
    "motorisation_par_habitant",
    "lots_stationnement_par_habitant",
    "lots_stationnement_par_copro",
]].describe()

## Construction du score

Strategie:
- normalisation robuste par rang percentil (evite l'effet des tres grosses villes),
- 3 composantes ponderees,
- legere penalite sur les communes a faible completude des donnees,
- legere penalite pour profils tres ruraux (faible population + faible copropriete).


In [ ]:
def percentile_rank(series: pd.Series) -> pd.Series:
    s = series.fillna(series.median())
    return s.rank(method="average", pct=True)

features_weights = {
    # Demande / pression
    "population_totale": 0.20,
    "nb_vehicules": 0.15,
    "motorisation_par_habitant": 0.05,
    # Opportunite copro
    "nb_copros": 0.15,
    "nb_logements": 0.10,
    "grandes_copros_nb": 0.10,
    "part_grandes_copros": 0.05,
    # Gisement parking
    "total_lots_stationnement": 0.12,
    "nb_points_parking": 0.05,
    "moyenne_lots_par_point": 0.03,
}

score_df = df.copy()

for feat in features_weights:
    score_df[f"pr_{feat}"] = percentile_rank(score_df[feat])

score_df["score_demande"] = (
    0.50 * score_df["pr_population_totale"]
    + 0.375 * score_df["pr_nb_vehicules"]
    + 0.125 * score_df["pr_motorisation_par_habitant"]
)

score_df["score_copro"] = (
    0.375 * score_df["pr_nb_copros"]
    + 0.25 * score_df["pr_nb_logements"]
    + 0.25 * score_df["pr_grandes_copros_nb"]
    + 0.125 * score_df["pr_part_grandes_copros"]
)

score_df["score_parking"] = (
    0.60 * score_df["pr_total_lots_stationnement"]
    + 0.25 * score_df["pr_nb_points_parking"]
    + 0.15 * score_df["pr_moyenne_lots_par_point"]
)

score_df["score_base_0_100"] = 100 * (
    0.40 * score_df["score_demande"]
    + 0.40 * score_df["score_copro"]
    + 0.20 * score_df["score_parking"]
)

# completude des variables critiques (max impact: -30%)
critical_features = list(features_weights.keys())
score_df["data_completeness"] = score_df[critical_features].notna().mean(axis=1)
score_df["coef_completeness"] = 0.70 + 0.30 * score_df["data_completeness"]

# penalite rurale legere: peu de population et peu de copros
rural_flag = (
    (score_df["pr_population_totale"] < 0.20)
    & (score_df["pr_nb_copros"] < 0.20)
)
score_df["coef_ruralite"] = np.where(rural_flag, 0.85, 1.00)

score_df["score_potentiel"] = (
    score_df["score_base_0_100"]
    * score_df["coef_completeness"]
    * score_df["coef_ruralite"]
).round(2)

score_df["rang_potentiel"] = score_df["score_potentiel"].rank(ascending=False, method="min").astype(int)

q80 = score_df["score_potentiel"].quantile(0.80)
q60 = score_df["score_potentiel"].quantile(0.60)
q40 = score_df["score_potentiel"].quantile(0.40)

score_df["priorite"] = np.select(
    [
        score_df["score_potentiel"] >= q80,
        score_df["score_potentiel"] >= q60,
        score_df["score_potentiel"] >= q40,
    ],
    ["A - tres prioritaire", "B - prioritaire", "C - secondaire"],
    default="D - faible priorite",
)

score_df[["score_potentiel", "score_demande", "score_copro", "score_parking"]].describe()

In [ ]:
preview_cols = [
    "code_insee", "code_postal", "nom_commune", "nom_dep",
    "score_potentiel", "rang_potentiel", "priorite",
    "score_demande", "score_copro", "score_parking",
    "population_totale", "nb_vehicules", "nb_copros", "nb_logements",
    "total_lots_stationnement", "nb_points_parking", "lat", "long"
]

score_df[preview_cols].sort_values("score_potentiel", ascending=False).head(20)

In [ ]:
kpi_cols = [
    "code_insee", "code_postal", "nom_commune", "nom_dep", "lat", "long",
    "score_potentiel", "rang_potentiel", "priorite",
    "score_demande", "score_copro", "score_parking",
    "data_completeness", "coef_completeness", "coef_ruralite",
    "population_totale", "nb_vehicules", "nb_vp_elec",
    "nb_copros", "nb_logements", "grandes_copros_nb", "part_grandes_copros",
    "total_lots_stationnement", "moyenne_lots_par_point", "nb_points_parking",
]

kpi_df = score_df[kpi_cols].sort_values("score_potentiel", ascending=False).reset_index(drop=True)
kpi_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print(f"Fichier KPI exporte: {OUTPUT_PATH}")
print(f"Nombre de lignes: {len(kpi_df)}")
kpi_df.head(10)